<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers torch sacrebleu evaluate nltk -q

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")
print("GPU available:", torch.cuda.is_available())

Setup done!
GPU available: True


In [ ]:
from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

dataset = load_dataset("bentrevett/multi30k")

print(dataset)
print("\n--- Sample record ---")
print("English:", dataset["train"][0]["en"])
print("German: ", dataset["train"][0]["de"])
print(f"\nTrain: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'de'],
        num_rows: 29000
    })
    validation: Dataset({
        features: ['en', 'de'],
        num_rows: 1014
    })
    test: Dataset({
        features: ['en', 'de'],
        num_rows: 1000
    })
})

--- Sample record ---
English: Two young, White males are outside near many bushes.
German:  Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.

Train: 29000, Val: 1014, Test: 1000


In [ ]:
from collections import Counter

# Build vocabularies
def build_vocab(sentences, max_vocab=10000):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.lower().split())
    vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
    for word, _ in counter.most_common(max_vocab - 4):
        vocab[word] = len(vocab)
    return vocab

train_en = [ex["en"] for ex in dataset["train"]]
train_de = [ex["de"] for ex in dataset["train"]]
val_en   = [ex["en"] for ex in dataset["validation"]]
val_de   = [ex["de"] for ex in dataset["validation"]]
test_en  = [ex["en"] for ex in dataset["test"]]
test_de  = [ex["de"] for ex in dataset["test"]]

src_vocab = build_vocab(train_en)
tgt_vocab = build_vocab(train_de)
tgt_idx2word = {v: k for k, v in tgt_vocab.items()}

print(f"Source (EN) vocab size: {len(src_vocab)}")
print(f"Target (DE) vocab size: {len(tgt_vocab)}")

# Encode sentences
def encode(sentence, vocab, max_len=50):
    tokens = sentence.lower().split()[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids = [2] + ids + [3]  # <SOS> + tokens + <EOS>
    ids += [0] * (max_len + 2 - len(ids))  # pad
    return ids

print("\nSample encoding:")
print("EN:", train_en[0])
print("Encoded:", encode(train_en[0], src_vocab)[:10], "...")

Source (EN) vocab size: 10000
Target (DE) vocab size: 10000

Sample encoding:
EN: Two young, White males are outside near many bushes.
Encoded: [2, 13, 1347, 22, 824, 15, 63, 72, 167, 1648] ...


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src = [encode(s, src_vocab, max_len) for s in src_sentences]
        self.tgt = [encode(s, tgt_vocab, max_len) for s in tgt_sentences]

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.src[idx], dtype=torch.long),
            torch.tensor(self.tgt[idx], dtype=torch.long)
        )

train_dataset = TranslationDataset(train_en, train_de, src_vocab, tgt_vocab)
val_dataset   = TranslationDataset(val_en,   val_de,   src_vocab, tgt_vocab)
test_dataset  = TranslationDataset(test_en,  test_de,  src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64)
test_loader  = DataLoader(test_dataset,  batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Datasets ready!")
print(f"Train batches: {len(train_loader)}")

Datasets ready!
Train batches: 454


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, hidden = self.rnn(embedded)
        # Combine bidirectional hidden states
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        return F.softmax(self.v(energy).squeeze(2), dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = Attention(hidden_dim)
        self.rnn = nn.GRU(embed_dim + hidden_dim * 2, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim * 3 + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden, encoder_outputs):
        tgt = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))
        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(1)
        context = torch.bmm(attn_weights, encoder_outputs)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=2).squeeze(1))
        return prediction, hidden.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        tgt_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        encoder_outputs, hidden = self.encoder(src)
        dec_input = tgt[:, 0]

        for t in range(1, tgt_len):
            output, hidden = self.decoder(dec_input, hidden, encoder_outputs)
            outputs[:, t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            dec_input = tgt[:, t] if teacher_force else output.argmax(1)

        return outputs

# Initialize model
EMBED_DIM = 256
HIDDEN_DIM = 512

encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM)
seq2seq = Seq2Seq(encoder, decoder, device).to(device)

total_params = sum(p.numel() for p in seq2seq.parameters())
print(f"Seq2Seq model parameters: {total_params:,}")

Seq2Seq model parameters: 29,483,280


In [ ]:
optimizer_seq2seq = torch.optim.Adam(seq2seq.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def train_seq2seq(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src, tgt)
        output = output[:, 1:].reshape(-1, output.shape[-1])
        tgt = tgt[:, 1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_seq2seq(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt, teacher_forcing_ratio=0)
            output = output[:, 1:].reshape(-1, output.shape[-1])
            tgt = tgt[:, 1:].reshape(-1)
            loss = criterion(output, tgt)
            total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 10
print("Training Seq2Seq + Attention...")
for epoch in range(EPOCHS):
    train_loss = train_seq2seq(seq2seq, train_loader, optimizer_seq2seq, criterion, device)
    val_loss = evaluate_seq2seq(seq2seq, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("\nDone!")

Training Seq2Seq + Attention...
Epoch 1/10 | Train Loss: 2.9532 | Val Loss: 3.3282
Epoch 2/10 | Train Loss: 2.2218 | Val Loss: 3.2694
Epoch 3/10 | Train Loss: 1.8001 | Val Loss: 3.3479
Epoch 4/10 | Train Loss: 1.5807 | Val Loss: 3.4815
Epoch 5/10 | Train Loss: 1.4370 | Val Loss: 3.5359
Epoch 6/10 | Train Loss: 1.3047 | Val Loss: 3.6774
Epoch 7/10 | Train Loss: 1.2155 | Val Loss: 3.7251
Epoch 8/10 | Train Loss: 1.1418 | Val Loss: 3.8673
Epoch 9/10 | Train Loss: 1.0675 | Val Loss: 3.8802
Epoch 10/10 | Train Loss: 1.0078 | Val Loss: 4.0262

Done!


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

print("Loading Helsinki-NLP model (EN→DE)...")
model_name = "Helsinki-NLP/opus-mt-en-de"
helsinki_tokenizer = MarianTokenizer.from_pretrained(model_name)
helsinki_model = MarianMTModel.from_pretrained(model_name).to(device)
helsinki_model.eval()

def helsinki_translate(sentences, batch_size=32):
    translations = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inputs = helsinki_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = helsinki_model.generate(**inputs)
        decoded = helsinki_tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(decoded)
    return translations

# Test
print("\n--- Helsinki Sample ---")
sample = test_en[0]
print("EN:", sample)
print("DE (Helsinki):", helsinki_translate([sample])[0])
print("DE (Reference):", test_de[0])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading Helsinki-NLP model (EN→DE)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


--- Helsinki Sample ---
EN: A man in an orange hat starring at something.
DE (Helsinki): Ein Mann in einem orangenen Hut, der mit etwas zu tun hat.
DE (Reference): Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.


In [ ]:
# Seq2Seq translation function
def seq2seq_translate(sentences, model, src_vocab, tgt_idx2word, max_len=50):
    model.eval()
    translations = []
    with torch.no_grad():
        for sent in sentences:
            src = torch.tensor([encode(sent, src_vocab, max_len)], dtype=torch.long).to(device)
            encoder_outputs, hidden = model.encoder(src)
            dec_input = torch.tensor([tgt_vocab["<SOS>"]], dtype=torch.long).to(device)
            result = []
            for _ in range(max_len):
                output, hidden = model.decoder(dec_input, hidden, encoder_outputs)
                top1 = output.argmax(1)
                word = tgt_idx2word.get(top1.item(), "<UNK>")
                if word == "<EOS>":
                    break
                if word not in ("<PAD>", "<SOS>", "<UNK>"):
                    result.append(word)
                dec_input = top1
            translations.append(" ".join(result))
    return translations

# Generate translations on test set
print("Generating Seq2Seq translations...")
seq2seq_preds = seq2seq_translate(test_en, seq2seq, src_vocab, tgt_idx2word)

print("Generating Helsinki translations...")
helsinki_preds = helsinki_translate(test_en)

references = test_de
print("\nDone! Sample outputs:")
print("EN        :", test_en[0])
print("Reference :", references[0])
print("Seq2Seq   :", seq2seq_preds[0])
print("Helsinki  :", helsinki_preds[0])

Generating Seq2Seq translations...
Generating Helsinki translations...

Done! Sample outputs:
EN        : A man in an orange hat starring at something.
Reference : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Seq2Seq   : ein mann mit einem orangefarbenen sicherheitsweste starrt etwas.
Helsinki  : Ein Mann in einem orangenen Hut, der mit etwas zu tun hat.


In [ ]:
!pip install bert_score -q

import evaluate

sacrebleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")
chrf = evaluate.load("chrf")
bertscore = evaluate.load("bertscore")

print("Computing BLEU...")
bleu_seq2seq = sacrebleu.compute(predictions=seq2seq_preds, references=[[r] for r in references])
bleu_helsinki = sacrebleu.compute(predictions=helsinki_preds, references=[[r] for r in references])

print("Computing METEOR...")
meteor_seq2seq = meteor.compute(predictions=seq2seq_preds, references=references)
meteor_helsinki = meteor.compute(predictions=helsinki_preds, references=references)

print("Computing ChrF...")
chrf_seq2seq = chrf.compute(predictions=seq2seq_preds, references=[[r] for r in references])
chrf_helsinki = chrf.compute(predictions=helsinki_preds, references=[[r] for r in references])

print("Computing BERTScore...")
bertscore_seq2seq = bertscore.compute(predictions=seq2seq_preds, references=references, lang="de")
bertscore_helsinki = bertscore.compute(predictions=helsinki_preds, references=references, lang="de")

print("\n" + "="*70)
print(f"{'Metric':<20} {'Seq2Seq+Attn':>20} {'Helsinki':>20}")
print("="*70)
print(f"{'BLEU':<20} {bleu_seq2seq['score']:>20.4f} {bleu_helsinki['score']:>20.4f}")
print(f"{'METEOR':<20} {meteor_seq2seq['meteor']:>20.4f} {meteor_helsinki['meteor']:>20.4f}")
print(f"{'ChrF':<20} {chrf_seq2seq['score']:>20.4f} {chrf_helsinki['score']:>20.4f}")
print(f"{'BERTScore F1':<20} {sum(bertscore_seq2seq['f1'])/len(bertscore_seq2seq['f1']):>20.4f} {sum(bertscore_helsinki['f1'])/len(bertscore_helsinki['f1']):>20.4f}")
print("="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Computing BLEU...
Computing METEOR...
Computing ChrF...
Computing BERTScore...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


Metric                       Seq2Seq+Attn             Helsinki
BLEU                               4.7366              36.2524
METEOR                             0.5131               0.6668
ChrF                              40.2865              64.2841
BERTScore F1                       0.7524               0.8967


In [ ]:
print("="*80)
print("QUALITATIVE EXAMPLES")
print("="*80)

for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"EN (Source)  : {test_en[i]}")
    print(f"Reference    : {references[i]}")
    print(f"Seq2Seq+Attn : {seq2seq_preds[i]}")
    print(f"Helsinki     : {helsinki_preds[i]}")
    print("="*80)

QUALITATIVE EXAMPLES

--- Example 1 ---
EN (Source)  : A man in an orange hat starring at something.
Reference    : Ein Mann mit einem orangefarbenen Hut, der etwas anstarrt.
Seq2Seq+Attn : ein mann mit einem orangefarbenen sicherheitsweste starrt etwas.
Helsinki     : Ein Mann in einem orangenen Hut, der mit etwas zu tun hat.

--- Example 2 ---
EN (Source)  : A Boston Terrier is running on lush green grass in front of a white fence.
Reference    : Ein Boston Terrier läuft über saftig-grünes Gras vor einem weißen Zaun.
Seq2Seq+Attn : ein läuft vor einem grünen wiese grünen wiese vor gras.
Helsinki     : Ein Boston Terrier läuft auf üppig grünem Gras vor einem weißen Zaun.

--- Example 3 ---
EN (Source)  : A girl in karate uniform breaking a stick with a front kick.
Reference    : Ein Mädchen in einem Karateanzug bricht ein Brett mit einem Tritt.
Seq2Seq+Attn : ein mädchen in karateanzügen uniform mit einem einen
Helsinki     : Ein Mädchen in Karate-Uniform bricht einen Stock mit einem 